# 🤖 04 — Robot Brain: Predicting the Next Action

> **Goal:** show the agent a single photo of a workspace and ask *"what should the robot do
> next?"* — the core question in robotics.

---

## Why this matters

A robot sees the world through a camera. To act, it must look at **one image** and decide
the next move. Cosmos-Reason2 is trained on robot data, so it's good at exactly this.

```mermaid
flowchart LR
    IMG["camera view<br/>(one photo)"]:::data
    M["CosmosVisionModel<br/>reasoning=True"]:::reason
    T["&lt;think&gt; what objects?<br/>what's reachable?<br/>what's the goal? &lt;/think&gt;"]:::think
    ACT(["Reach for the cup<br/>and grasp the handle"]):::out
    IMG --> M --> T --> ACT
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])            # import strands_cosmos from repo root

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.path.join(PROJECT_ROOT, "sample.mp4")
SAMPLE_IMAGE = os.path.join(PROJECT_ROOT, "sample.png")

def gpu_ready() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

READY = gpu_ready()
print("🎮 GPU available :", READY)
if READY:
    import torch
    print("   Device       :", torch.cuda.get_device_name(0))
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   (CPU-only is fine for reading along — the heavy cells will skip safely.)")
print("📹 sample.mp4   :", os.path.exists(SAMPLE_VIDEO))
print("🖼  sample.png   :", os.path.exists(SAMPLE_IMAGE))

## Step 1 — Look at what the robot sees
We'll display the sample image right in the notebook (no GPU needed for this part).

In [ ]:
if os.path.exists(SAMPLE_IMAGE):
    from PIL import Image
    from IPython.display import display
    img = Image.open(SAMPLE_IMAGE)
    print(f"🖼  {img.size[0]}x{img.size[1]} pixels")
    display(img.resize((640, int(640 * img.size[1] / img.size[0]))))
else:
    print("⏭  sample.png not found.")

## Step 2 — Load the vision + thinking model

In [ ]:
agent = None
if READY and os.path.exists(SAMPLE_IMAGE):
    from strands import Agent
    from strands_cosmos import CosmosVisionModel
    t0 = time.time()
    model = CosmosVisionModel(
        model_id="nvidia/Cosmos-Reason2-2B",
        reasoning=True,
        params={"max_tokens": 2048, "temperature": 0.6},
    )
    agent = Agent(model=model)
    print(f"✅ Ready in {time.time()-t0:.1f}s")
else:
    print("⏭  Need a GPU + sample.png.")

## Step 3 — The classic question
Note the **`<image>...</image>`** tag (images use `image`, videos use `video`).

In [ ]:
if agent is not None:
    t0 = time.time()
    agent(f"<image>{SAMPLE_IMAGE}</image> What can be the next immediate action?")
    print(f"\n⏱  {time.time()-t0:.1f}s")
else:
    print("⏭  Skipped.")

## Step 4 — Understand the scene in detail
Ask about objects and what a robot arm could grab.

In [ ]:
if agent is not None:
    agent(
        f"<image>{SAMPLE_IMAGE}</image> "
        "Describe all objects, their spatial relationships, and what a robot arm could manipulate."
    )
else:
    print("⏭  Skipped.")

# 🎓 What you learned

| Concept | Takeaway |
|---------|----------|
| `<image>path</image>` | Attach a **still image** (vs. `<video>`) |
| Next-action prediction | The core loop of embodied AI / robotics |
| Reasoning + vision | Combine eyes **and** step-by-step thinking |

**Next:** [05 — Tool Usage](05_tool_usage.ipynb) — use Cosmos as a **building block** inside
any agent. 🧰 →